In [ ]:
import json
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

import config
from src.db_io import leer_tabla_sqlite
from src.log_decisiones import registrar_decision

fact = leer_tabla_sqlite(config.ORO_DB, "fact_cliente_score")
df = leer_tabla_sqlite(config.ORO_DB, "cliente_features")
datos = fact.merge(df[["numero_id", "desc_segmento"]], on="numero_id", how="left")
(config.OUTPUTS_DIR / "powerbi").mkdir(parents=True, exist_ok=True)
(config.OUTPUTS_DIR / "eda").mkdir(parents=True, exist_ok=True)

por_nivel = (
    datos.groupby(["poblacion", "nivel"], as_index=False)
    .agg(n_clientes=("numero_id", "count"))
    .sort_values(["poblacion", "nivel"])
)
print("Clientes por nivel de prioridad y población:")
print(por_nivel.pivot(index="nivel", columns="poblacion", values="n_clientes").to_string())


In [ ]:
# SPEC_V2 §7: monto potencial agregado por nivel, en los tres escenarios,
# SOLO para la población con historial. La población sin historial no tiene
# monto: su monto es NULL, no cero (§6.3).
con_hist = datos[datos["tiene_historial_inversion"] == 1]

montos = (
    con_hist.groupby("nivel", as_index=False)
    .agg(n_clientes=("numero_id", "count"),
         monto_conservador=("monto_conservador_12m", "sum"),
         monto_base=("monto_base_12m", "sum"),
         monto_optimista=("monto_optimista_12m", "sum"),
         # D5: descomposición además del total.
         monto_app_base=("monto_app_12m_base", "sum"),
         monto_prod_conservadores_base=("monto_prod_conservadores_12m_base", "sum"))
    .sort_values("nivel")
)
print("Monto potencial agregado a 12 meses (solo población con historial de inversión):")
print(montos.to_string(index=False))
print(f"\nClientes SIN historial de inversión (monto NULL, no cero): "
      f"{int((datos['tiene_historial_inversion'] == 0).sum()):,}")
print(
    "\nNOTA: estas sumas por nivel heredan la misma limitación estadística que "
    "el agregado total (ver el AVISO completo, con evidencia medida, en la "
    "celda del resumen ejecutivo más abajo): son la suma del límite p10/p90 de "
    "CADA cliente, no una banda de incertidumbre agregada válida."
)


In [ ]:
por_segmento = (
    datos.groupby(["desc_segmento", "poblacion", "nivel"], as_index=False)
    .agg(n_clientes=("numero_id", "count"))
)
tabla_seg = por_segmento.pivot_table(
    index="desc_segmento", columns="nivel", values="n_clientes",
    aggfunc="sum", fill_value=0)
print("Distribución de niveles por desc_segmento:")
print(tabla_seg.to_string())


In [ ]:
dimensionamiento = (
    datos.groupby(["nivel", "poblacion", "desc_segmento"], as_index=False)
    .agg(n_clientes=("numero_id", "count"),
         monto_conservador=("monto_conservador_12m", "sum"),
         monto_base=("monto_base_12m", "sum"),
         monto_optimista=("monto_optimista_12m", "sum"),
         # D5: descomposición del monto base (además del total) por componente.
         monto_app_base=("monto_app_12m_base", "sum"),
         monto_prod_conservadores_base=("monto_prod_conservadores_12m_base", "sum"),
         score_medio=("score", "mean"))
    .sort_values(["nivel", "poblacion", "desc_segmento"])
)
dimensionamiento.to_csv(config.OUTPUTS_DIR / "powerbi" / "dimensionamiento.csv", index=False)

priorizados = datos[datos["nivel"].isin(["A", "B"])]
resumen = {
    "n_clientes_total": int(len(datos)),
    "n_clientes_priorizados_A_B": int(len(priorizados)),
    "n_nivel_A": int((datos["nivel"] == "A").sum()),
    "n_nivel_A_con_historial": int(((datos["nivel"] == "A") &
                                    (datos["poblacion"] == "con_historial")).sum()),
    "n_nivel_A_sin_historial": int(((datos["nivel"] == "A") &
                                    (datos["poblacion"] == "sin_historial")).sum()),
    "oportunidad_12m_conservador": float(con_hist["monto_conservador_12m"].sum()),
    "oportunidad_12m_base": float(con_hist["monto_base_12m"].sum()),
    "oportunidad_12m_optimista": float(con_hist["monto_optimista_12m"].sum()),
    "n_clientes_con_monto": int(con_hist["monto_base_12m"].notna().sum()),
}

# AVISO (hallazgo del coordinador, decisión de negocio ABIERTA, no resuelta
# aquí): el agregado [conservador, optimista] de arriba se construye sumando
# el límite p10/p90 de CADA cliente por separado. Eso equivale a asumir que
# TODOS los clientes con historial caen simultáneamente en su propio escenario
# más pesimista (o más optimista) -- ignora la diversificación entre clientes.
# Si los errores de backtest fueran razonablemente independientes entre
# clientes, la incertidumbre agregada escalaría con sqrt(n), no con n. Se mide
# la evidencia directamente en vez de asumirla:
n_clientes_agregado = int(con_hist["monto_base_12m"].notna().sum())
ancho_medio_cliente = float(
    (con_hist["monto_optimista_12m"] - con_hist["monto_conservador_12m"]).mean())
ancho_agregado = resumen["oportunidad_12m_optimista"] - resumen["oportunidad_12m_conservador"]
factor_escala_medido = (
    ancho_agregado / ancho_medio_cliente if ancho_medio_cliente else float("nan"))
raiz_n = float(np.sqrt(n_clientes_agregado))

resumen["aviso_agregado_ingenuo"] = {
    "metodo": "suma_de_limites_p10_p90_por_cliente",
    "supuesto_implicito": "errores_de_backtest_perfectamente_correlacionados_entre_clientes",
    "n_clientes": n_clientes_agregado,
    "raiz_n": raiz_n,
    "factor_escala_medido_ancho_agregado_sobre_ancho_medio_cliente": factor_escala_medido,
    "recomendacion": (
        "citar_unicamente_base_como_titular; "
        "NO_citar_conservador_ni_optimista_agregado_sin_este_aviso"
    ),
}

with open(config.OUTPUTS_DIR / "eda" / "resumen_ejecutivo.json", "w", encoding="utf-8") as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)

print("=" * 78)
print("RESUMEN EJECUTIVO")
print("=" * 78)
print(f"Clientes totales scoreados:            {resumen['n_clientes_total']:,}")
print(f"Clientes priorizados (niveles A y B):  {resumen['n_clientes_priorizados_A_B']:,}")
print(f"  · nivel A con historial:             {resumen['n_nivel_A_con_historial']:,}")
print(f"  · nivel A sin historial (lookalike): {resumen['n_nivel_A_sin_historial']:,}")
print(f"\nRango de oportunidad a 12 meses (población con historial, "
      f"{resumen['n_clientes_con_monto']:,} clientes):")
print(f"  conservador: {resumen['oportunidad_12m_conservador']:,.0f}")
print(f"  base:        {resumen['oportunidad_12m_base']:,.0f}")
print(f"  optimista:   {resumen['oportunidad_12m_optimista']:,.0f}")
print(
    "\nADVERTENCIAS DE INTERPRETACIÓN:\n"
    "· La cifra es un RANGO, no un pronóstico puntual: el horizonte de 12 meses se "
    "extrapola desde ~13 meses de historia, validado solo contra 3 meses (§6.3). El "
    "escenario 'base' está recentrado por la MEDIANA del error de backtest (el modelo "
    "sobre-predice sistemáticamente) y el rango [conservador, optimista] usa los "
    "percentiles 10/90 del error, no 25/75 (ver DECISIONES.md, clave=metrica_error_monto, "
    "y notebooks/06_monto_12m.ipynb) — la cifra 'base' de este notebook es la versión YA "
    "corregida por sesgo, no la predicción cruda del modelo.\n"
    "· Los clientes 'A' sin historial se rankean por SIMILITUD (lookalike), no por "
    "probabilidad validada: en ese segmento la etiqueta es 0 por construcción (§6.1).\n"
    "· Los niveles NO son comparables entre poblaciones: cada 'A' es el 25% superior "
    "de SU población (§6.2)."
)

print(
    "\n" + "=" * 78 + "\n"
    "AVISO — EL RANGO AGREGADO [CONSERVADOR, OPTIMISTA] NO ES UNA BANDA DE\n"
    "INCERTIDUMBRE VÁLIDA. DECISIÓN DE NEGOCIO ABIERTA, NO RESUELTA AQUÍ.\n"
    + "=" * 78 + "\n"
    f"El rango de este resumen se construye sumando el límite p10/p90 de CADA "
    f"cliente por separado (n={n_clientes_agregado:,}). Eso equivale a asumir "
    f"que los {n_clientes_agregado:,} clientes caen SIMULTÁNEAMENTE en su "
    "propio escenario más pesimista (o más optimista) — ignora por completo la "
    "diversificación entre clientes.\n"
    f"Evidencia medida: ancho del agregado / ancho medio por cliente = "
    f"{factor_escala_medido:,.1f} (aprox. n = {n_clientes_agregado:,}). Si los "
    f"errores de backtest fueran razonablemente independientes entre clientes, "
    f"la incertidumbre agregada escalaría con sqrt(n) = {raiz_n:,.0f}, órdenes "
    "de magnitud más angosta que la banda mostrada arriba.\n"
    f"CONCLUSIÓN: la única cifra agregada defendible como titular ante el "
    f"negocio es 'base' ({resumen['oportunidad_12m_base']:,.0f} COP). NO citar "
    f"'conservador' ({resumen['oportunidad_12m_conservador']:,.0f} COP) ni "
    f"'optimista' ({resumen['oportunidad_12m_optimista']:,.0f} COP) agregados "
    "frente al negocio sin este aviso adjunto. Si 'conservador' sale negativo, "
    "NUNCA presentarlo como 'la oportunidad pesimista' sin esta explicación: un "
    "lector vería una cifra negativa de billones de pesos y concluiría, "
    "razonablemente, que el modelo está roto -- el problema es el método de "
    "agregación, no el modelo (los escenarios POR CLIENTE, de donde sale esta "
    "suma, sí son correctos: ver notebooks/06_monto_12m.ipynb).\n"
    "Este notebook NO inventa un método de agregación alternativo (p.ej. "
    "propagación por sqrt(n) o una simulación de la correlación real entre "
    "errores de cliente): es una decisión de negocio pendiente, registrada en "
    "el log de decisiones y en el README, no resuelta en silencio aquí."
)

registrar_decision(
    clave="agregacion_rango_oportunidad_12m",
    decision="reportar_base_como_titular_marcar_conservador_optimista_agregado_como_no_citable",
    motivo=(
        f"El agregado [conservador, optimista] de SPEC_V2 §7 se construye sumando "
        f"el límite p10/p90 de cada cliente ({n_clientes_agregado:,} clientes con "
        f"historial de inversión en esta corrida), lo que asume errores de backtest "
        "perfectamente correlacionados entre TODOS ellos. Bajo independencia "
        f"razonable la incertidumbre agregada escalaría con sqrt(n) = {raiz_n:,.0f}, "
        f"no con n = {n_clientes_agregado:,}: la banda mostrada está sobrestimada en "
        "órdenes de magnitud (evidencia medida en esta misma celda, no asumida: "
        f"ancho_agregado / ancho_medio_cliente = {factor_escala_medido:,.1f}). No se "
        "inventa un método de agregación alternativo en este notebook -- es una "
        "decisión de negocio pendiente. Mientras no se decida, 'base' es la única "
        "cifra agregada defendible como titular; 'conservador'/'optimista' "
        "agregados NO deben citarse frente al negocio sin este aviso."
    ),
    evidencia={
        "n_clientes": n_clientes_agregado,
        "raiz_n": raiz_n,
        "factor_escala_medido": factor_escala_medido,
        "oportunidad_12m_conservador": resumen["oportunidad_12m_conservador"],
        "oportunidad_12m_base": resumen["oportunidad_12m_base"],
        "oportunidad_12m_optimista": resumen["oportunidad_12m_optimista"],
    },
)
